In [1]:
# ================================================================
# Cell 1 — Frozen N=8 benchmark specification
#
# This cell contains benchmark configuration only.
# The physical model is the same frozen Stage-4 synthetic system.
# ================================================================

from pathlib import Path
from collections import Counter, defaultdict

import gzip
import hashlib
import inspect
import json
import pickle
import shutil
import time

import numpy as np
import pandas as pd
import sympy as sp


# --------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------

SEED = 20260811
DATA_SEED = SEED + 202

weight_rng = np.random.default_rng(SEED)
data_rng = np.random.default_rng(DATA_SEED)


# --------------------------------------------------------------
# Paths
# --------------------------------------------------------------

DATA_DIR = Path("./stage4_n8_data")
RUN_DIR = Path("./stage4_n8_reference_v34")

DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

LEARNER_DATA_FILE = DATA_DIR / "stage4_n8_learner_data.npz"
DATA_MANIFEST_FILE = DATA_DIR / "dataset_manifest.json"
GENERATION_LEDGER_FILE = DATA_DIR / "generation_ledger.csv"
ORACLE_CONTEXT_FILE = DATA_DIR / "stage4_n8_benchmark_context.pkl.gz"

CHECKPOINT_DIR = RUN_DIR / "checkpoint"
PROVENANCE_DIR = RUN_DIR / "provenance"
RESULT_DIR = RUN_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------
# System / model
# --------------------------------------------------------------

N = 8
nodes = tuple(range(1, N + 1))

x_symbols = sp.symbols(
    f"x1:{N + 1}"
)

LAMBDA = 0.5
LAMBDA_EXACT = sp.Rational(1, 2)


# --------------------------------------------------------------
# Dataset configuration
# --------------------------------------------------------------

EPS_VALUES = np.array([
    0.005,
    0.0075,
    0.010,
    0.015,
    0.020,
    0.030,
    0.040,
    0.060,
    0.080,
    0.100,
    0.120,
], dtype=float)

N_TRAIN = 3000
N_TEST = 1000
N_TOTAL = N_TRAIN + N_TEST

MAX_RK4_STEP = 1e-3


# --------------------------------------------------------------
# Learner model class
# --------------------------------------------------------------

MAX_POLY_DEGREE = 3
MAX_SUPPORT_SIZE = 4


print("Frozen N=8 benchmark initialized.")
print("N =", N)
print("seed =", SEED)
print("data seed =", DATA_SEED)
print("train / test =", N_TRAIN, "/", N_TEST)
print("epsilon values =", EPS_VALUES)
print(
    "generic learner class:",
    f"degree <= {MAX_POLY_DEGREE},",
    f"support <= {MAX_SUPPORT_SIZE}",
)


Frozen N=8 benchmark initialized.
N = 8
seed = 20260811
data seed = 20261013
train / test = 3000 / 1000
epsilon values = [0.005  0.0075 0.01   0.015  0.02   0.03   0.04   0.06   0.08   0.1
 0.12  ]
generic learner class: degree <= 3, support <= 4


In [2]:
# ================================================================
# Cell 2 — Frozen microscopic network and temporal protocol
# ================================================================

# --------------------------------------------------------------
# Microscopic pairwise graph
# --------------------------------------------------------------

microscopic_edges = (
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),
    (6, 7),
    (7, 8),
    (8, 1),
    (1, 3),
    (2, 4),
    (3, 5),
    (5, 7),
)

assert len(microscopic_edges) == 12
assert len(set(microscopic_edges)) == 12


# --------------------------------------------------------------
# Frozen heterogeneous edge weights
#
# Same construction as the original Stage 4 benchmark:
# integer weights sampled once on [800, 1200], then divided by 1000.
# --------------------------------------------------------------

edge_weight_integer = {
    edge: int(weight_rng.integers(800, 1201))
    for edge in microscopic_edges
}

edge_weight = {
    edge: value / 1000.0
    for edge, value in edge_weight_integer.items()
}

edge_weight_exact = {
    edge: sp.Rational(value, 1000)
    for edge, value in edge_weight_integer.items()
}


# --------------------------------------------------------------
# Frozen six-snapshot temporal protocol
#
# Each microscopic edge appears exactly twice over the full
# six-snapshot protocol.
# --------------------------------------------------------------

snapshots = (
    ((1, 2), (3, 4), (5, 6), (7, 8)),
    ((2, 3), (4, 5), (6, 7), (8, 1)),
    ((1, 3), (2, 4), (3, 5), (5, 7)),
    ((1, 2), (4, 5), (7, 8), (3, 5)),
    ((2, 3), (5, 6), (8, 1), (5, 7)),
    ((3, 4), (6, 7), (1, 3), (2, 4)),
)


# --------------------------------------------------------------
# Consistency audit
# --------------------------------------------------------------

snapshot_edge_counts = Counter(
    edge
    for snapshot in snapshots
    for edge in snapshot
)

assert len(snapshots) == 6
assert all(len(snapshot) == 4 for snapshot in snapshots)

assert set(snapshot_edge_counts) == set(microscopic_edges)

assert all(
    snapshot_edge_counts[edge] == 2
    for edge in microscopic_edges
)


# --------------------------------------------------------------
# Compact summary
# --------------------------------------------------------------

print("Frozen microscopic network:")
print("  nodes =", N)
print("  edges =", len(microscopic_edges))
print("  snapshots =", len(snapshots))
print("  edges per snapshot =", len(snapshots[0]))
print("  exposure per microscopic edge = 2")

print()
print("Frozen edge weights:")
for edge in microscopic_edges:
    print(
        f"  {edge}: {edge_weight[edge]:.3f}"
    )

print()
print("Temporal protocol:")
for r, snapshot in enumerate(snapshots, start=1):
    print(f"  G{r} = {snapshot}")

Frozen microscopic network:
  nodes = 8
  edges = 12
  snapshots = 6
  edges per snapshot = 4
  exposure per microscopic edge = 2

Frozen edge weights:
  (1, 2): 1.006
  (2, 3): 0.911
  (3, 4): 1.105
  (4, 5): 0.917
  (5, 6): 1.177
  (6, 7): 1.119
  (7, 8): 0.917
  (8, 1): 0.944
  (1, 3): 1.026
  (2, 4): 1.002
  (3, 5): 1.067
  (5, 7): 1.031

Temporal protocol:
  G1 = ((1, 2), (3, 4), (5, 6), (7, 8))
  G2 = ((2, 3), (4, 5), (6, 7), (8, 1))
  G3 = ((1, 3), (2, 4), (3, 5), (5, 7))
  G4 = ((1, 2), (4, 5), (7, 8), (3, 5))
  G5 = ((2, 3), (5, 6), (8, 1), (5, 7))
  G6 = ((3, 4), (6, 7), (1, 3), (2, 4))


In [3]:
# ================================================================
# Cell 3 — Microscopic vector fields and snapshot generators
#
# Pairwise temporal dynamics:
#     phi(x) = x + 0.5 x^2
#
# Persistent native triads:
#     H_123 and H_258
#     g = 0.02
#
# Both symbolic and numerical/vectorized implementations are
# defined from the same frozen physical model.
# ================================================================


# --------------------------------------------------------------
# Nonlinear pairwise interaction
# --------------------------------------------------------------

def phi_symbolic(z):
    return z + LAMBDA_EXACT * z**2


def phi_numeric(z):
    return z + LAMBDA * z**2


def edge_field_symbolic(edge):
    """
    Conservative nonlinear pairwise vector field for one edge.

    For edge (i,j):
        F_i = w_ij [phi(x_j) - phi(x_i)]
        F_j = -F_i
    """
    i, j = edge

    if edge not in edge_weight_exact:
        raise KeyError(f"Unknown microscopic edge: {edge}")

    w = edge_weight_exact[edge]

    field = sp.zeros(N, 1)

    interaction = w * (
        phi_symbolic(x_symbols[j - 1])
        -
        phi_symbolic(x_symbols[i - 1])
    )

    field[i - 1] += interaction
    field[j - 1] -= interaction

    return field


def edge_field_numeric(X, edge):
    """
    Vectorized numerical version.

    Parameters
    ----------
    X : ndarray, shape (..., N)
        One state or a batch of states.
    edge : tuple
        Frozen microscopic edge.

    Returns
    -------
    F : ndarray, same shape as X
    """
    i, j = edge

    if edge not in edge_weight:
        raise KeyError(f"Unknown microscopic edge: {edge}")

    w = edge_weight[edge]

    F = np.zeros_like(X, dtype=float)

    interaction = w * (
        phi_numeric(X[..., j - 1])
        -
        phi_numeric(X[..., i - 1])
    )

    F[..., i - 1] += interaction
    F[..., j - 1] -= interaction

    return F


# --------------------------------------------------------------
# Persistent native triadic interactions
#
# For h = {i,j,k},
#
# T_i =
#     g x_j x_k
#     - g/2 x_i x_j
#     - g/2 x_i x_k
#
# with cyclic permutations for j and k.
#
# This field is:
#   - quadratic
#   - symmetric under permutation of the three nodes
#   - conservative
#   - irreducibly triadic
# --------------------------------------------------------------

NATIVE_TRIADS = (
    (1, 2, 3),
    (2, 5, 8),
)

NATIVE_G = 0.02
NATIVE_G_EXACT = sp.Rational(1, 50)


def triad_field_symbolic(triad, g=NATIVE_G_EXACT):
    i, j, k = triad

    xi = x_symbols[i - 1]
    xj = x_symbols[j - 1]
    xk = x_symbols[k - 1]

    field = sp.zeros(N, 1)

    field[i - 1] += (
        g * xj * xk
        - g / 2 * xi * xj
        - g / 2 * xi * xk
    )

    field[j - 1] += (
        g * xi * xk
        - g / 2 * xj * xi
        - g / 2 * xj * xk
    )

    field[k - 1] += (
        g * xi * xj
        - g / 2 * xk * xi
        - g / 2 * xk * xj
    )

    return field


def triad_field_numeric(X, triad, g=NATIVE_G):
    """
    Vectorized native triadic field.
    """
    i, j, k = triad

    xi = X[..., i - 1]
    xj = X[..., j - 1]
    xk = X[..., k - 1]

    F = np.zeros_like(X, dtype=float)

    F[..., i - 1] += (
        g * xj * xk
        - 0.5 * g * xi * xj
        - 0.5 * g * xi * xk
    )

    F[..., j - 1] += (
        g * xi * xk
        - 0.5 * g * xj * xi
        - 0.5 * g * xj * xk
    )

    F[..., k - 1] += (
        g * xi * xj
        - 0.5 * g * xk * xi
        - 0.5 * g * xk * xj
    )

    return F


# --------------------------------------------------------------
# Precompute symbolic microscopic fields
# --------------------------------------------------------------

edge_fields_symbolic = {
    edge: edge_field_symbolic(edge)
    for edge in microscopic_edges
}

native_triad_fields_symbolic = {
    triad: triad_field_symbolic(triad)
    for triad in NATIVE_TRIADS
}


# --------------------------------------------------------------
# Basic physical consistency checks
# --------------------------------------------------------------

# Every microscopic field should conserve sum_i x_i.
for edge, field in edge_fields_symbolic.items():
    assert sp.simplify(sum(field)) == 0

for triad, field in native_triad_fields_symbolic.items():
    assert sp.simplify(sum(field)) == 0


# Numerical vectorization check
X_check = data_rng.uniform(
    -0.5,
    0.5,
    size=(16, N)
)

for edge in microscopic_edges:
    F_check = edge_field_numeric(X_check, edge)

    assert F_check.shape == X_check.shape

    assert np.max(
        np.abs(F_check.sum(axis=1))
    ) < 1e-14


for triad in NATIVE_TRIADS:
    F_check = triad_field_numeric(X_check, triad)

    assert F_check.shape == X_check.shape

    assert np.max(
        np.abs(F_check.sum(axis=1))
    ) < 1e-14


print("Microscopic vector fields initialized.")
print("  pairwise edge fields =", len(edge_fields_symbolic))
print("  persistent native triads =", NATIVE_TRIADS)
print("  native triad strength =", NATIVE_G)
print("  symbolic conservation: PASS")
print("  numerical conservation: PASS")

# ================================================================
# Forward temporal generators
#
# Each snapshot contains:
#   4 temporally activated pairwise edge fields
#   + the same two persistent native triadic fields
#
# No reverse protocol is constructed anywhere in this notebook.
# ================================================================


# --------------------------------------------------------------
# Persistent native contribution
# --------------------------------------------------------------

native_total_symbolic = sum(
    native_triad_fields_symbolic.values(),
    sp.zeros(N, 1)
)


def native_total_numeric(X):
    F = np.zeros_like(X, dtype=float)

    for triad in NATIVE_TRIADS:
        F += triad_field_numeric(
            X,
            triad
        )

    return F


# --------------------------------------------------------------
# Symbolic snapshot generators
# --------------------------------------------------------------

snapshot_fields_symbolic = []


for snapshot in snapshots:

    pairwise_part = sum(
        (
            edge_fields_symbolic[edge]
            for edge in snapshot
        ),
        sp.zeros(N, 1)
    )

    full_field = pairwise_part + native_total_symbolic

    snapshot_fields_symbolic.append(
        sp.Matrix([
            sp.expand(component)
            for component in full_field
        ])
    )


# --------------------------------------------------------------
# Numerical/vectorized snapshot generator
# --------------------------------------------------------------

def snapshot_field_numeric(X, snapshot_index):
    """
    Evaluate one full snapshot generator on one state or a batch.

    Parameters
    ----------
    X : ndarray, shape (..., N)
    snapshot_index : int
        Python index 0,...,5.
    """

    snapshot = snapshots[snapshot_index]

    F = np.zeros_like(X, dtype=float)

    # Temporally activated pairwise sector
    for edge in snapshot:
        F += edge_field_numeric(
            X,
            edge
        )

    # Persistent native triads
    F += native_total_numeric(X)

    return F


# --------------------------------------------------------------
# Physical consistency audit
# --------------------------------------------------------------

assert len(snapshot_fields_symbolic) == len(snapshots)


# Symbolic conservation
for r, field in enumerate(
    snapshot_fields_symbolic,
    start=1
):
    assert sp.simplify(sum(field)) == 0


# Numerical conservation
X_check = data_rng.uniform(
    -0.5,
    0.5,
    size=(32, N)
)

max_conservation_error = 0.0

for r in range(len(snapshots)):

    F_check = snapshot_field_numeric(
        X_check,
        r
    )

    error = np.max(
        np.abs(
            F_check.sum(axis=1)
        )
    )

    max_conservation_error = max(
        max_conservation_error,
        error
    )


assert max_conservation_error < 1e-13


# --------------------------------------------------------------
# Persistent-exposure check
#
# Native HOIs are present in every snapshot, so their exposure
# over a full aggregation window is epsilon, not 6 epsilon.
# --------------------------------------------------------------

n_snapshots = len(snapshots)
snapshot_fraction = 1.0 / n_snapshots

assert n_snapshots == 6
assert np.isclose(
    n_snapshots * snapshot_fraction,
    1.0
)


print("Forward snapshot generators constructed.")
print("  number of snapshots =", n_snapshots)
print("  pairwise edges / snapshot =", 4)
print("  persistent native triads / snapshot =", len(NATIVE_TRIADS))
print("  snapshot duration = epsilon /", n_snapshots)
print(
    "  max numerical conservation error =",
    f"{max_conservation_error:.3e}"
)
print("  reverse protocol constructed = False")

Microscopic vector fields initialized.
  pairwise edge fields = 12
  persistent native triads = ((1, 2, 3), (2, 5, 8))
  native triad strength = 0.02
  symbolic conservation: PASS
  numerical conservation: PASS
Forward snapshot generators constructed.
  number of snapshots = 6
  pairwise edges / snapshot = 4
  persistent native triads / snapshot = 2
  snapshot duration = epsilon / 6
  max numerical conservation error = 3.331e-16
  reverse protocol constructed = False


In [4]:
# ================================================================
# Cell 4 — Exact symbolic oracle construction
# ================================================================

from collections import Counter
import sympy as sp

# --------------------------------------------------------------
# 1. Exact symbolic F^(0)
#
# Six equal-duration snapshots:
#
#     F^(0) = (1/6) sum_r G_r
# --------------------------------------------------------------

F0_oracle_symbolic = sum(
    snapshot_fields_symbolic,
    sp.zeros(N, 1)
) / len(snapshot_fields_symbolic)

F0_oracle_symbolic = sp.Matrix([
    sp.expand(expr)
    for expr in F0_oracle_symbolic
])


# --------------------------------------------------------------
# 2. Exact symbolic F^(1)
#
# Chronological protocol:
#
#     G1 -> G2 -> ... -> G6
#
# with each snapshot duration epsilon / 6.
#
# For the effective generator:
#
#     F_eff = F^(0) + epsilon F^(1) + O(epsilon^2)
#
# the BCH first-order correction is
#
#     F^(1) = (1 / (2 * 6^2)) sum_{s>r} [G_s, G_r]
#           = (1 / 72) sum_{s>r} [G_s, G_r]
#
# Overall Lie-bracket sign does not affect support recovery.
# --------------------------------------------------------------

def lie_bracket(F, G):
    """
    Vector-field Lie bracket.

        [F, G] = J_G F - J_F G

    Swapping the convention only flips the global sign of F^(1),
    so the structural support oracle is unchanged.
    """
    JF = F.jacobian(x_symbols)
    JG = G.jacobian(x_symbols)

    return sp.Matrix([
        sp.expand(expr)
        for expr in (JG * F - JF * G)
    ])


F1_oracle_symbolic = sp.zeros(N, 1)

for s in range(len(snapshot_fields_symbolic)):
    for r in range(s):
        F1_oracle_symbolic += lie_bracket(
            snapshot_fields_symbolic[r],
            snapshot_fields_symbolic[s],
        )

F1_oracle_symbolic /= (
    2 * len(snapshot_fields_symbolic)**2
)

F1_oracle_symbolic = sp.Matrix([
    sp.expand(expr)
    for expr in F1_oracle_symbolic
])


# --------------------------------------------------------------
# 3. Extract minimal structural supports
#
# For a monomial appearing in output i:
#
#     support = {output i} U {variables appearing in monomial}
#
# Node labels are returned 1-based, exactly matching TSC_AGLASSO.
# --------------------------------------------------------------

def symbolic_structural_supports(field):
    supports = set()

    for output_index, expr in enumerate(field):

        poly = sp.Poly(
            sp.expand(expr),
            *x_symbols
        )

        for powers, coefficient in poly.terms():

            # Exact symbolic zero check
            if sp.simplify(coefficient) == 0:
                continue

            variables = {
                j + 1
                for j, power in enumerate(powers)
                if power > 0
            }

            support = tuple(sorted(
                variables | {output_index + 1}
            ))

            supports.add(support)

    return tuple(sorted(
        supports,
        key=lambda s: (len(s), s)
    ))


oracle_F0 = symbolic_structural_supports(
    F0_oracle_symbolic
)

oracle_F1 = symbolic_structural_supports(
    F1_oracle_symbolic
)




# --------------------------------------------------------------
# Frozen oracle checks
# --------------------------------------------------------------

oracle_count_F0 = Counter(
    len(s)
    for s in oracle_F0
)

oracle_count_F1 = Counter(
    len(s)
    for s in oracle_F1
)

assert len(oracle_F0) == 25
assert oracle_count_F0 == {
    1: 8,
    2: 15,
    3: 2,
}

assert len(oracle_F1) == 58
assert oracle_count_F1 == {
    1: 3,
    2: 25,
    3: 25,
    4: 5,
}


# --------------------------------------------------------------
# Persist truth-side benchmark context
#
# This file is NOT loaded by the learner.
# It is reopened only after blind inference is complete.
# --------------------------------------------------------------

oracle_context = {
    "dataset":
        "Stage-4 N=8 forward-only TSC benchmark",

    "N":
        N,

    "seed":
        SEED,

    "data_seed":
        DATA_SEED,

    "lambda_nonlinear":
        LAMBDA,

    "microscopic_edges":
        microscopic_edges,

    "edge_weight_integer":
        edge_weight_integer,

    "snapshots":
        snapshots,

    "native_triads":
        NATIVE_TRIADS,

    "native_g":
        NATIVE_G,

    "F0_oracle_symbolic":
        F0_oracle_symbolic,

    "F1_oracle_symbolic":
        F1_oracle_symbolic,

    "oracle_supports_F0":
        oracle_F0,

    "oracle_supports_F1":
        oracle_F1,
}

with gzip.open(
    ORACLE_CONTEXT_FILE,
    "wb",
) as f:
    pickle.dump(
        oracle_context,
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


print("Exact symbolic oracle constructed.")
print(
    "  F0 supports =",
    len(oracle_F0),
    "| by size =",
    dict(sorted(oracle_count_F0.items())),
)
print(
    "  F1 supports =",
    len(oracle_F1),
    "| by size =",
    dict(sorted(oracle_count_F1.items())),
)
print(
    "  oracle context saved:",
    ORACLE_CONTEXT_FILE.resolve(),
)


Exact symbolic oracle constructed.
  F0 supports = 25 | by size = {1: 8, 2: 15, 3: 2}
  F1 supports = 58 | by size = {1: 3, 2: 25, 3: 25, 4: 5}
  oracle context saved: C:\Users\liu.xuanc\Desktop\Code\TSC-TemporalStructureClosure\stage4_n8_data\stage4_n8_benchmark_context.pkl.gz


In [5]:
# ================================================================
# Cell 5 — Forward endpoint generation + persisted learner dataset
#
# Observations:
#     X0
#     XF(epsilon)
#
# Same initial conditions are used for every epsilon.
# No reverse trajectories are generated.
# ================================================================


# --------------------------------------------------------------
# Dataset RNG
#
# Reinitialize explicitly so previous diagnostic random draws
# cannot change the formal dataset.
# --------------------------------------------------------------

DATA_SEED = SEED + 202
dataset_rng = np.random.default_rng(DATA_SEED)


# --------------------------------------------------------------
# Initial conditions
# --------------------------------------------------------------

X0 = dataset_rng.uniform(
    -0.5,
    0.5,
    size=(N_TOTAL, N)
)

X0_train = X0[:N_TRAIN]
X0_test = X0[N_TRAIN:]


# --------------------------------------------------------------
# Vectorized fixed-step RK4
# --------------------------------------------------------------

def rk4_integrate_snapshot(
    X,
    snapshot_index,
    duration,
    max_step=MAX_RK4_STEP,
):
    """
    Integrate one snapshot generator for a given duration.

    X can contain a batch of states with shape (n_samples, N).
    """

    n_steps = max(
        1,
        int(np.ceil(duration / max_step))
    )

    dt = duration / n_steps

    Y = np.array(
        X,
        dtype=float,
        copy=True
    )

    for _ in range(n_steps):

        k1 = snapshot_field_numeric(
            Y,
            snapshot_index
        )

        k2 = snapshot_field_numeric(
            Y + 0.5 * dt * k1,
            snapshot_index
        )

        k3 = snapshot_field_numeric(
            Y + 0.5 * dt * k2,
            snapshot_index
        )

        k4 = snapshot_field_numeric(
            Y + dt * k3,
            snapshot_index
        )

        Y += (
            dt / 6.0
        ) * (
            k1
            + 2.0 * k2
            + 2.0 * k3
            + k4
        )

    return Y


def forward_protocol(
    X_initial,
    epsilon,
):
    """
    Apply the six snapshots in chronological order.

        G1 -> G2 -> ... -> G6

    The full aggregation-window duration is epsilon.
    Each snapshot lasts epsilon / 6.
    """

    Y = np.array(
        X_initial,
        dtype=float,
        copy=True
    )

    snapshot_duration = (
        epsilon
        /
        len(snapshots)
    )

    for snapshot_index in range(
        len(snapshots)
    ):

        Y = rk4_integrate_snapshot(
            Y,
            snapshot_index,
            snapshot_duration,
        )

    return Y


# --------------------------------------------------------------
# Generate forward endpoint observations
# --------------------------------------------------------------

forward_data = {}

generation_rows = []


initial_mass = X0.sum(axis=1)


for epsilon in EPS_VALUES:

    XF = forward_protocol(
        X0,
        epsilon
    )

    forward_data[
        float(epsilon)
    ] = XF


    # ----------------------------------------------------------
    # Basic sanity diagnostics
    # ----------------------------------------------------------

    mass_error = np.max(
        np.abs(
            XF.sum(axis=1)
            -
            initial_mass
        )
    )

    displacement_rms = np.sqrt(
        np.mean(
            (XF - X0) ** 2
        )
    )

    generation_rows.append({
        "epsilon":
            float(epsilon),

        "displacement_rms":
            float(displacement_rms),

        "max_conservation_error":
            float(mass_error),

        "state_min":
            float(XF.min()),

        "state_max":
            float(XF.max()),
    })


generation_ledger = pd.DataFrame(
    generation_rows
)




# --------------------------------------------------------------
# Stack endpoint observations in the same format as N=50
#
# XF_* shape:
#     (n_epsilon, n_samples, N)
# --------------------------------------------------------------

XF_train = np.stack([
    forward_data[
        float(epsilon)
    ][:N_TRAIN]

    for epsilon in EPS_VALUES
], axis=0)


XF_test = np.stack([
    forward_data[
        float(epsilon)
    ][N_TRAIN:]

    for epsilon in EPS_VALUES
], axis=0)


# --------------------------------------------------------------
# Semantic learner-data SHA-256
# --------------------------------------------------------------

def _update_semantic_hash(
    hasher,
    name,
    array,
):
    arr = np.ascontiguousarray(
        np.asarray(array)
    )

    hasher.update(
        name.encode("utf-8")
    )

    hasher.update(
        str(arr.dtype).encode("utf-8")
    )

    hasher.update(
        np.asarray(
            arr.shape,
            dtype=np.int64,
        ).tobytes()
    )

    hasher.update(
        arr.view(
            np.uint8
        ).tobytes()
    )


h = hashlib.sha256()

for name, array in [
    ("X0_train", X0_train),
    ("XF_train", XF_train),
    ("X0_test", X0_test),
    ("XF_test", XF_test),
    ("epsilon_values", EPS_VALUES),
]:
    _update_semantic_hash(
        h,
        name,
        array,
    )

LEARNER_DATA_SHA256 = h.hexdigest()


# --------------------------------------------------------------
# Persist learner-only endpoint data
# --------------------------------------------------------------

np.savez_compressed(
    LEARNER_DATA_FILE,
    X0_train=X0_train,
    XF_train=XF_train,
    X0_test=X0_test,
    XF_test=XF_test,
    epsilon_values=EPS_VALUES,
)


# --------------------------------------------------------------
# Graph / protocol fingerprint
# --------------------------------------------------------------

graph_payload = {
    "edges":
        [list(edge) for edge in microscopic_edges],

    "edge_weight_integer":
        {
            str(edge): int(value)
            for edge, value in edge_weight_integer.items()
        },

    "snapshots":
        [
            [list(edge) for edge in snapshot]
            for snapshot in snapshots
        ],

    "native_triads":
        [list(t) for t in NATIVE_TRIADS],
}

GRAPH_FINGERPRINT = hashlib.sha256(
    json.dumps(
        graph_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:12]


# --------------------------------------------------------------
# Persist manifest and generation ledger
# --------------------------------------------------------------

DATA_MANIFEST = {
    "dataset":
        "Stage-4 N=8 forward-only TSC benchmark",

    "learner_data_sha256":
        LEARNER_DATA_SHA256,

    "graph_fingerprint":
        GRAPH_FINGERPRINT,

    "N":
        N,

    "n_microscopic_edges":
        len(microscopic_edges),

    "n_snapshots":
        len(snapshots),

    "lambda_nonlinear":
        LAMBDA,

    "native_triads":
        [list(t) for t in NATIVE_TRIADS],

    "native_g":
        NATIVE_G,

    "n_train":
        N_TRAIN,

    "n_test":
        N_TEST,

    "epsilon_values":
        EPS_VALUES.tolist(),

    "RK4_max_step":
        MAX_RK4_STEP,

    "seed_weights":
        SEED,

    "seed_data":
        DATA_SEED,
}

with open(
    DATA_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        DATA_MANIFEST,
        f,
        indent=2,
    )

generation_ledger.to_csv(
    GENERATION_LEDGER_FILE,
    index=False,
)


# --------------------------------------------------------------
# Final generation checks
# --------------------------------------------------------------

assert X0_train.shape == (
    N_TRAIN,
    N,
)

assert X0_test.shape == (
    N_TEST,
    N,
)

assert XF_train.shape == (
    len(EPS_VALUES),
    N_TRAIN,
    N,
)

assert XF_test.shape == (
    len(EPS_VALUES),
    N_TEST,
    N,
)

assert generation_ledger[
    "max_conservation_error"
].max() < 1e-12


print("Forward-only learner dataset persisted.")
print("  train X0 =", X0_train.shape)
print("  train XF =", XF_train.shape)
print("  test X0  =", X0_test.shape)
print("  test XF  =", XF_test.shape)
print("  learner SHA256 =", LEARNER_DATA_SHA256)
print("  graph fingerprint =", GRAPH_FINGERPRINT)
print("  reverse trajectories generated = False")
print()
display(generation_ledger)


Forward-only learner dataset persisted.
  train X0 = (3000, 8)
  train XF = (11, 3000, 8)
  test X0  = (1000, 8)
  test XF  = (11, 1000, 8)
  learner SHA256 = 068795c4c3c891c3d57eff2826d20851e8889111cbdd43cb2c0f3e29a690f74b
  graph fingerprint = e166e9e86e7a
  reverse trajectories generated = False



,epsilon,displacement_rms,max_conservation_error,state_min,state_max
0,0.0050,0.001756,4.440892e-16,-0.499023,0.498703
1,0.0075,0.002629,4.440892e-16,-0.498586,0.498155
2,0.0100,0.003498,8.881784e-16,-0.498439,0.497608
3,0.0150,0.005226,6.661338e-16,-0.498210,0.496933
4,0.0200,0.006940,6.661338e-16,-0.497980,0.496361
5,0.0300,0.010326,8.881784e-16,-0.497698,0.495214
6,0.0400,0.013658,8.881784e-16,-0.497481,0.494142
7,0.0600,0.020164,1.110223e-15,-0.497018,0.492671
8,0.0800,0.026465,1.554312e-15,-0.496518,0.491173
9,0.1000,0.032568,1.332268e-15,-0.495981,0.489650


In [6]:
# ================================================================
# Cell 6 — Reload persisted learner data / blind boundary
#
# From this point until the oracle-reveal cell, inference uses ONLY:
#
#     X0_train
#     XF_train(epsilon)
#     X0_test
#     XF_test(epsilon)
#     epsilon
#
# No graph, weights, snapshots, native triads, or oracle supports
# are passed to TSCInference.
# ================================================================

with np.load(
    LEARNER_DATA_FILE
) as data:

    inference_X0_train = np.asarray(
        data["X0_train"],
        dtype=float,
    )

    inference_XF_train = np.asarray(
        data["XF_train"],
        dtype=float,
    )

    inference_X0_test = np.asarray(
        data["X0_test"],
        dtype=float,
    )

    inference_XF_test = np.asarray(
        data["XF_test"],
        dtype=float,
    )

    inference_eps = np.asarray(
        data["epsilon_values"],
        dtype=float,
    )


with open(
    DATA_MANIFEST_FILE,
    "r",
    encoding="utf-8",
) as f:
    DATA_MANIFEST = json.load(f)


# --------------------------------------------------------------
# Recompute semantic SHA-256 from learner-visible data
# --------------------------------------------------------------

h = hashlib.sha256()

for name, array in [
    ("X0_train", inference_X0_train),
    ("XF_train", inference_XF_train),
    ("X0_test", inference_X0_test),
    ("XF_test", inference_XF_test),
    ("epsilon_values", inference_eps),
]:
    _update_semantic_hash(
        h,
        name,
        array,
    )

RELOADED_SHA256 = h.hexdigest()

assert (
    RELOADED_SHA256
    ==
    DATA_MANIFEST[
        "learner_data_sha256"
    ]
)


print("Blind learner dataset reloaded.")
print(
    "  train shapes =",
    inference_X0_train.shape,
    inference_XF_train.shape,
)
print(
    "  test shapes  =",
    inference_X0_test.shape,
    inference_XF_test.shape,
)
print(
    "  learner-data SHA256 verified:",
    RELOADED_SHA256,
)


Blind learner dataset reloaded.
  train shapes = (3000, 8) (11, 3000, 8)
  test shapes  = (1000, 8) (11, 1000, 8)
  learner-data SHA256 verified: 068795c4c3c891c3d57eff2826d20851e8889111cbdd43cb2c0f3e29a690f74b


In [7]:
# ================================================================
# Cell 7 — Blind TSC inference with TSC_AGLASSO v3.4
#
# Same public learner class as the N=50 benchmark.
# The learner receives endpoint observations only.
# ================================================================

import TSC_AGLASSO as tsc_module
from TSC_AGLASSO import TSCInference


# --------------------------------------------------------------
# Verify implementation
# --------------------------------------------------------------

TSC_VERSION = getattr(
    tsc_module,
    "_IMPLEMENTATION_VERSION",
    "unknown",
)

print(
    "TSC_AGLASSO implementation =",
    TSC_VERSION,
)

assert TSC_VERSION == "v3.5"


# --------------------------------------------------------------
# Freeze exact source used
# --------------------------------------------------------------

TSC_SOURCE_FILE = Path(
    inspect.getfile(
        tsc_module
    )
)


def file_sha256(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


TSC_SOURCE_SHA256 = file_sha256(
    TSC_SOURCE_FILE
)

TSC_SOURCE_COPY = (
    PROVENANCE_DIR
    /
    "TSC_AGLASSO_v3.5_used.py"
)

shutil.copy2(
    TSC_SOURCE_FILE,
    TSC_SOURCE_COPY,
)


print(
    "TSC source =",
    TSC_SOURCE_FILE.resolve(),
)

print(
    "TSC source SHA256 =",
    TSC_SOURCE_SHA256,
)


# --------------------------------------------------------------
# Generic blind learner
#
# k_max = 4 intentionally allows the complete frozen N=8 oracle
# hierarchy without giving the learner its actual support set.
# --------------------------------------------------------------

model = TSCInference(
    max_interaction_order=4,
    max_polynomial_degree=3,
    temporal_order=1,
    verbose=True,
)


fit_start = time.perf_counter()


result = model.fit(
    inference_X0_train,
    inference_XF_train,
    inference_eps,

    X0_test=
        inference_X0_test,

    XF_test=
        inference_XF_test,

    checkpoint_dir=
        CHECKPOINT_DIR,

    resume_from_checkpoint=
        True,
)


fit_runtime_seconds = (
    time.perf_counter()
    -
    fit_start
)


print()
print("=" * 64)
print("N=8 blind TSC inference complete")
print("=" * 64)

print(
    "runtime [s] =",
    f"{fit_runtime_seconds:.3f}",
)

print(
    "runtime [min] =",
    f"{fit_runtime_seconds / 60.0:.3f}",
)

print(
    "checkpoint directory =",
    CHECKPOINT_DIR.resolve(),
)

print()

result.summary(
    show_supports=False
)


TSC_AGLASSO implementation = v3.5
TSC source = C:\Users\liu.xuanc\Desktop\Code\TSC-TemporalStructureClosure\TSC_AGLASSO.py
TSC source SHA256 = 1863bd6697c5f6b1bea8035bb4322f48ff3e0e03f550d4ed2433f8efbc14a0d6
TSC configuration:
  max_interaction_order = 4
  max_polynomial_degree = 3
  temporal_order = 1
  extrapolation eps = [0.005  0.0075 0.01   0.015 ]
  library features/output = 164
  structural groups = 162

  F0 checkpoint version mismatch; ignoring (v3.4 != v3.5).
Starting KKT working-set Adaptive Group LASSO for F^(0) (KKT-event-driven validation path)...
  internal split = 2400 fit / 600 validation states
  adaptive path <= 40 lambda evaluations | geometric ratio=0.750 | entry fraction=0.980
  maximum KKT jump = 2.00 decades | emergency floor = 1e-10 * lambda_max
  local refinement <= 4 lambdas around an interior minimum validation-loss adaptive point
  selector = validation one-standard-error rule (largest admissible lambda)
  pilot = adaptive Ridge | alpha range = [6.713e-12, 

In [11]:
import gzip
import pickle

with gzip.open(
    ORACLE_CONTEXT_FILE,
    "rb",
) as f:
    oracle_context = pickle.load(f)

oracle = set(
    tuple(s)
    for s in oracle_context["oracle_supports_F1"]
)

ledger = result.F1.path_ledger

for n_groups in [57, 58, 59]:

    rows = ledger[
        ledger["n_groups"] == n_groups
    ]

    if len(rows) == 0:
        continue

    row = rows.loc[
        rows["validation_relative_error"].idxmin()
    ]

    inferred = set(
        tuple(s)
        for s in row["active_supports"]
    )

    TP = inferred & oracle
    FP = inferred - oracle
    FN = oracle - inferred

    print()
    print("=" * 50)
    print("groups =", n_groups)
    print("stage =", row["path_stage"])
    print("lambda =", f"{row['lambda']:.6e}")
    print(
        "val_rel =",
        f"{row['validation_relative_error']:.6e}",
    )
    print(
        "TP / FP / FN =",
        len(TP),
        len(FP),
        len(FN),
    )
    print("FP =", sorted(FP))
    print("FN =", sorted(FN))


groups = 57
stage = adaptive
lambda = 5.248339e-11
val_rel = 1.966705e-05
TP / FP / FN = 57 0 1
FP = []
FN = [(1,)]

groups = 58
stage = adaptive
lambda = 3.781805e-13
val_rel = 9.076797e-06
TP / FP / FN = 58 0 0
FP = []
FN = []

groups = 59
stage = adaptive
lambda = 8.486983e-14
val_rel = 4.538035e-06
TP / FP / FN = 58 1 0
FP = [(5,)]
FN = []


In [ ]:
# ================================================================
# Cell 8 — Save blind result + compact structural output
#
# Still no oracle reveal in this cell.
# ================================================================

structure_table = result.structure_table()

display(
    structure_table
)


# --------------------------------------------------------------
# Persist blind result and diagnostics
# --------------------------------------------------------------

RESULT_PICKLE = (
    RESULT_DIR
    /
    "TSC_result_v34.pkl.gz"
)

with gzip.open(
    RESULT_PICKLE,
    "wb",
) as f:
    pickle.dump(
        result,
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )


structure_table.to_csv(
    RESULT_DIR
    /
    "structure_table.csv",
    index=False,
)

result.F0.path_ledger.to_csv(
    RESULT_DIR
    /
    "F0_path_ledger.csv",
    index=False,
)

result.F1.path_ledger.to_csv(
    RESULT_DIR
    /
    "F1_path_ledger.csv",
    index=False,
)

if result.F1.pruning_ledger is not None:
    result.F1.pruning_ledger.to_csv(
        RESULT_DIR
        /
        "F1_pruning_ledger.csv",
        index=False,
    )


blind_summary = {
    "implementation_version":
        TSC_VERSION,

    "algorithm_sha256":
        TSC_SOURCE_SHA256,

    "learner_data_sha256":
        RELOADED_SHA256,

    "runtime_seconds":
        float(
            fit_runtime_seconds
        ),

    "F0_groups":
        len(
            result.F0.selected_supports
        ),

    "F1_screening_groups":
        len(
            result.F1.screening_supports
        ),

    "F1_final_groups":
        len(
            result.F1.selected_supports
        ),

    "F0_test_relative_error":
        float(
            result.F0.test_relative_error
        ),

    "F1_test_relative_error":
        float(
            result.F1.test_relative_error
        ),

    "k_star":
        int(
            result.k_star
        ),
}

with open(
    RESULT_DIR
    /
    "blind_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        blind_summary,
        f,
        indent=2,
    )


print()
print("Blind result persisted.")
print(
    "  F0 groups =",
    len(
        result.F0.selected_supports
    ),
)
print(
    "  F1 screening groups =",
    len(
        result.F1.screening_supports
    ),
)
print(
    "  F1 final groups =",
    len(
        result.F1.selected_supports
    ),
)
print(
    "  k_star =",
    result.k_star,
)
print(
    "  result file =",
    RESULT_PICKLE.resolve(),
)


In [ ]:
# ================================================================
# Cell 9 — ORACLE REVEAL: structural + generator audit
#
# Oracle context is loaded only AFTER blind inference is complete.
# ================================================================

with gzip.open(
    ORACLE_CONTEXT_FILE,
    "rb",
) as f:
    oracle_context = pickle.load(f)


oracle_F0 = tuple(
    oracle_context[
        "oracle_supports_F0"
    ]
)

oracle_F1 = tuple(
    oracle_context[
        "oracle_supports_F1"
    ]
)

F0_oracle_symbolic = oracle_context[
    "F0_oracle_symbolic"
]

F1_oracle_symbolic = oracle_context[
    "F1_oracle_symbolic"
]


# --------------------------------------------------------------
# Structural comparison utility
# --------------------------------------------------------------

def compare_supports(
    oracle,
    inferred,
    label,
):
    oracle_set = set(
        oracle
    )

    inferred_set = set(
        inferred
    )

    TP = (
        oracle_set
        &
        inferred_set
    )

    FP = (
        inferred_set
        -
        oracle_set
    )

    FN = (
        oracle_set
        -
        inferred_set
    )

    precision = (
        len(TP)
        /
        len(inferred_set)
        if inferred_set
        else 1.0
    )

    recall = (
        len(TP)
        /
        len(oracle_set)
        if oracle_set
        else 1.0
    )

    f1_score = (
        2
        * precision
        * recall
        /
        (
            precision
            +
            recall
        )
        if (
            precision
            +
            recall
        ) > 0
        else 0.0
    )

    print("=" * 64)
    print(label)
    print("=" * 64)

    print(
        "Oracle groups by support size =",
        dict(
            sorted(
                Counter(
                    len(s)
                    for s in oracle_set
                ).items()
            )
        ),
    )

    print(
        "Inferred groups by support size =",
        dict(
            sorted(
                Counter(
                    len(s)
                    for s in inferred_set
                ).items()
            )
        ),
    )

    print()

    print(
        "Oracle supports   =",
        len(
            oracle_set
        ),
    )

    print(
        "Inferred supports =",
        len(
            inferred_set
        ),
    )

    print()

    print(
        "TP / FP / FN =",
        len(TP),
        "/",
        len(FP),
        "/",
        len(FN),
    )

    print(
        f"Precision = {precision:.6f}"
    )

    print(
        f"Recall    = {recall:.6f}"
    )

    print(
        f"F1 score  = {f1_score:.6f}"
    )

    print(
        "Exact support-set match =",
        oracle_set == inferred_set,
    )

    if FP:
        print()
        print("False positives:")
        for support in sorted(
            FP,
            key=lambda s: (
                len(s),
                s,
            ),
        ):
            print(
                " ",
                support,
            )

    if FN:
        print()
        print("False negatives:")
        for support in sorted(
            FN,
            key=lambda s: (
                len(s),
                s,
            ),
        ):
            print(
                " ",
                support,
            )

    return {
        "oracle":
            oracle_set,

        "inferred":
            inferred_set,

        "TP":
            TP,

        "FP":
            FP,

        "FN":
            FN,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1_score,
    }


# --------------------------------------------------------------
# Screening vs final-pruned F1
# --------------------------------------------------------------

audit_F0 = compare_supports(
    oracle_F0,
    result.F0.selected_supports,
    "F^(0) FINAL structural recovery",
)

print()

audit_F1_screen = compare_supports(
    oracle_F1,
    result.F1.screening_supports,
    "F^(1) AGLASSO SCREENING structural recovery",
)

print()

audit_F1_final = compare_supports(
    oracle_F1,
    result.F1.selected_supports,
    "F^(1) FINAL PRUNED structural recovery",
)


# --------------------------------------------------------------
# Exact symbolic oracle -> learner raw polynomial basis
# --------------------------------------------------------------

library = result._library

assert library is not None

exponent_to_feature = {
    tuple(
        exponent
    ):
        feature_index

    for feature_index, exponent in enumerate(
        library.exponents
    )
}


def symbolic_field_to_raw_coefficients(
    field_symbolic,
):
    B_raw = np.zeros(
        (
            len(
                library.exponents
            ),
            N,
        ),
        dtype=float,
    )

    exact_supports = set()

    for output_index in range(
        N
    ):
        polynomial = sp.Poly(
            sp.expand(
                field_symbolic[
                    output_index
                ]
            ),
            *x_symbols,
        )

        for exponent, coefficient in polynomial.terms():

            if sp.simplify(
                coefficient
            ) == 0:
                continue

            exponent = tuple(
                int(v)
                for v in exponent
            )

            if exponent not in exponent_to_feature:
                raise RuntimeError(
                    "Oracle term lies outside learner dictionary: "
                    f"output={output_index + 1}, "
                    f"exponent={exponent}, "
                    f"coefficient={coefficient}"
                )

            feature_index = exponent_to_feature[
                exponent
            ]

            B_raw[
                feature_index,
                output_index,
            ] = float(
                coefficient
            )

            variable_support = {
                j + 1

                for j, power in enumerate(
                    exponent
                )

                if power > 0
            }

            structural_support = tuple(
                sorted(
                    variable_support
                    |
                    {
                        output_index + 1
                    }
                )
            )

            exact_supports.add(
                structural_support
            )

    return (
        B_raw,
        tuple(
            sorted(
                exact_supports,
                key=lambda s: (
                    len(s),
                    s,
                ),
            )
        ),
    )


B0_oracle_raw, _ = symbolic_field_to_raw_coefficients(
    F0_oracle_symbolic
)

B1_oracle_raw, _ = symbolic_field_to_raw_coefficients(
    F1_oracle_symbolic
)


Theta_train_raw = library.evaluate_raw(
    inference_X0_train
)

Theta_test_raw = library.evaluate_raw(
    inference_X0_test
)


F0_oracle_train = (
    Theta_train_raw
    @
    B0_oracle_raw
)

F0_oracle_test = (
    Theta_test_raw
    @
    B0_oracle_raw
)

F1_oracle_train = (
    Theta_train_raw
    @
    B1_oracle_raw
)

F1_oracle_test = (
    Theta_test_raw
    @
    B1_oracle_raw
)


def relative_field_error(
    estimate,
    truth,
):
    return float(
        np.linalg.norm(
            estimate
            -
            truth
        )
        /
        np.linalg.norm(
            truth
        )
    )


print()
print("=" * 64)
print("EXACT GENERATOR AUDIT")
print("=" * 64)

print(
    "A_data vs oracle F0 train =",
    f"{relative_field_error(result.A_train, F0_oracle_train):.3e}",
)

print(
    "A_data vs oracle F0 test  =",
    f"{relative_field_error(result.A_test, F0_oracle_test):.3e}",
)

print(
    "F1_target vs oracle F1 train =",
    f"{relative_field_error(result.F1_target_train, F1_oracle_train):.3e}",
)

print(
    "F1_target vs oracle F1 test  =",
    f"{relative_field_error(result.F1_target_test, F1_oracle_test):.3e}",
)

print(
    "learned F0 vs oracle train =",
    f"{relative_field_error(Theta_train_raw @ result.F0.coefficients_raw, F0_oracle_train):.3e}",
)

print(
    "learned F0 vs oracle test  =",
    f"{relative_field_error(Theta_test_raw @ result.F0.coefficients_raw, F0_oracle_test):.3e}",
)

print(
    "learned F1 vs oracle train =",
    f"{relative_field_error(Theta_train_raw @ result.F1.coefficients_raw, F1_oracle_train):.3e}",
)

print(
    "learned F1 vs oracle test  =",
    f"{relative_field_error(Theta_test_raw @ result.F1.coefficients_raw, F1_oracle_test):.3e}",
)


# --------------------------------------------------------------
# Regression verdict
#
# These are REPORTING criteria, not learner inputs.
# They do not alter inference.
# --------------------------------------------------------------

regression_pass = bool(
    len(
        audit_F0[
            "FP"
        ]
    ) == 0
    and
    len(
        audit_F0[
            "FN"
        ]
    ) == 0
    and
    len(
        audit_F1_final[
            "FN"
        ]
    ) == 0
    and
    result.k_star == 4
)


print()
print("=" * 64)
print("FROZEN N=8 v3.4 REGRESSION VERDICT")
print("=" * 64)

print(
    "F0 exact support recovery =",
    (
        len(
            audit_F0[
                "FP"
            ]
        ) == 0
        and
        len(
            audit_F0[
                "FN"
            ]
        ) == 0
    ),
)

print(
    "F1 oracle recall =",
    f"{audit_F1_final['recall']:.6f}",
)

print(
    "F1 oracle precision =",
    f"{audit_F1_final['precision']:.6f}",
)

print(
    "F1 false positives =",
    len(
        audit_F1_final[
            "FP"
        ]
    ),
)

print(
    "F1 false negatives =",
    len(
        audit_F1_final[
            "FN"
        ]
    ),
)

print(
    "k_star =",
    result.k_star,
)

print(
    "REGRESSION PASS =",
    regression_pass,
)


In [ ]:
# ================================================================
# Cell 10 — v3.4 path / pruning diagnostics
#
# This cell is diagnostic only. It does not refit the model.
# ================================================================

F0_LEDGER_COLUMNS = [
    "path_stage",
    "stage_index",
    "lambda",
    "lambda_over_max",
    "n_groups",
    "n_parameters",
    "train_relative_error",
    "validation_relative_error",
    "working_set_size",
    "kkt_reactivations",
    "max_kkt_ratio",
    "final_kkt_satisfied",
]

F1_LEDGER_COLUMNS = [
    "path_stage",
    "stage_index",
    "lambda",
    "lambda_over_max",
    "n_groups",
    "n_parameters",
    "train_relative_error",
    "validation_relative_error",
    "working_set_size",
    "kkt_reactivations",
    "max_kkt_ratio",
    "final_kkt_satisfied",
]


print("=== F0 path ===")

display(
    result.F0.path_ledger[
        [
            c
            for c in F0_LEDGER_COLUMNS
            if c in result.F0.path_ledger.columns
        ]
    ]
)


print()
print("=== F1 AGLASSO path ===")

display(
    result.F1.path_ledger[
        [
            c
            for c in F1_LEDGER_COLUMNS
            if c in result.F1.path_ledger.columns
        ]
    ]
)


print()
print("=== F1 pruning path ===")

if result.F1.pruning_ledger is None:

    print(
        "No pruning ledger available."
    )

else:

    display(
        result.F1.pruning_ledger
    )


print()
print("=== v3.4 screening -> final ===")

print(
    "screening supports =",
    len(
        result.F1.screening_supports
    ),
)

print(
    "final supports =",
    len(
        result.F1.selected_supports
    ),
)

print(
    "pruned groups =",
    len(
        result.F1.screening_supports
    )
    -
    len(
        result.F1.selected_supports
    ),
)

print(
    "pruning threshold =",
    result.F1.pruning_threshold,
)

print(
    "base-path boundary hit =",
    result.F1.path_boundary_hit,
)

print(
    "path stages =",
    tuple(
        result.F1.path_ledger[
            "path_stage"
        ].drop_duplicates()
    ),
)


# --------------------------------------------------------------
# Readable sectors used in the original N=8 notebook
# --------------------------------------------------------------

target_supports = [
    (1, 2, 3),
    (2, 5, 8),
    (6, 8),
    (1, 2, 5, 8),
]


print()
print("=== Selected benchmark sectors ===")

display(
    result.structure_table()[
        result.structure_table()[
            "support"
        ].isin(
            target_supports
        )
    ]
)


for support in target_supports:

    if support in result._library.structural_groups:

        print()
        print(
            "support =",
            support,
        )

        if support in set(
            result.F0.selected_supports
        ):

            print(
                "F0 coefficients:"
            )

            display(
                result.sector_coefficients(
                    support,
                    temporal_order=0,
                    nonzero_only=True,
                )
            )

        if support in set(
            result.F1.selected_supports
        ):

            print(
                "F1 coefficients:"
            )

            display(
                result.sector_coefficients(
                    support,
                    temporal_order=1,
                    nonzero_only=True,
                )
            )
